# **Imports and Initalization**

In [50]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [51]:
!pip install pylatexenc

# **Macro Expander**

The job of this block is simple to state: read `custom_commands.tex` and use what's in it to replace shorthand macros with their full form. So `\vev{H}` becomes `\langle H \rangle`, `\zz` becomes `\mathbb{Z}`, and so on. That expanded version is what goes in the content field of each block.


In [52]:
"""
macro_expander.py
-----------------
Parses the custom_commands.tex and expands known macros in a
LaTeX string, producing a human-readable `content` field.

Handles:
  - \\def\\name{replacement}          (no-arg simple defs)
  - \\def\\name#1{replacement}         (single-arg \\newcommand style)
  - \\newcommand{\\name}[n]{replacement} (n-arg newcommand)
  - \\def\\name[#1,#2]{replacement}    (delimited-bracket args, e.g. \\identityN)

TikZ-heavy macros (those whose replacement body contains tikzpicture or
\\draw/\\node) are registered as STRIP targets — their call sites are
removed from content entirely.

Usage:
    expander = MacroExpander("config/custom_commands.tex")
    readable = expander.expand(raw_latex_string)
"""

import re
from dataclasses import dataclass, field
from typing import Optional


### The `data model` (`MacroDef`) is a small container that stores everything we know about one macro: its name, how many arguments it takes, what to replace it with, and whether it should be stripped entirely (TikZ macros) or expanded.


In [53]:
# ---------------------------------------------------------------------------
# Data model
# ---------------------------------------------------------------------------

@dataclass
class MacroDef:
    name: str           # without backslash
    num_args: int       # 0, 1, 2, ...
    replacement: str
    bracket_args: object  # False | True ([...] style) | 'paren' ((...) style)
    strip: bool = False # True → call site should be deleted, not expanded


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

_TIKZ_SIGNALS = re.compile(
    r'\\(begin\s*\{tikzpicture\}|draw|node|tikzpicture|tikzset|tikzstyle'
    r'|path|fill|scope)',
    re.IGNORECASE,
)

def _is_tikz_body(body: str) -> bool:
    return bool(_TIKZ_SIGNALS.search(body))


def _read_balanced_brace(s: str, start: int) -> tuple[str, int]:
    """Return (content_inside_braces, index_after_closing_brace)."""
    assert s[start] == '{', f"Expected '{{' at position {start}, got {s[start]!r}"
    depth = 0
    i = start
    while i < len(s):
        if s[i] == '{':
            depth += 1
        elif s[i] == '}':
            depth -= 1
            if depth == 0:
                return s[start + 1:i], i + 1
        i += 1
    raise ValueError(f"Unbalanced braces starting at {start} in: {s[start:start+40]!r}")


def _read_balanced_bracket(s: str, start: int) -> tuple[str, int]:
    """Return (content_inside_brackets, index_after_closing_bracket)."""
    assert s[start] == '[', f"Expected '[' at position {start}"
    depth = 0
    i = start
    while i < len(s):
        if s[i] == '[':
            depth += 1
        elif s[i] == ']':
            depth -= 1
            if depth == 0:
                return s[start + 1:i], i + 1
        i += 1
    raise ValueError(f"Unbalanced brackets starting at {start}")

### The `loader` (`_parse_defs` and `_parse_newcommands`) reads `custom_commands.tex` and populates a dictionary of `MacroDef` objects. The tricky part is that LaTeX has two ways to define macros and three ways to declare their arguments:
* `\newcommand{\vev}[1]{...}` — standard, args in `{curly braces}`
* `\def\identityN[#1,#2]{...}` — older style, args in `[square brackets]`
* `\def\nodeCS(#1,#2)(#3,#4,#5){...}` — the thesis's quiver macros, args in (parentheses)

### Each needs a different parser. For TikZ macros specifically, rather than trying to understand what they draw, we just check if the replacement body contains `\draw`, `\node`, `\tikzpicture` etc. — if yes, flag it for stripping.

### The `expander` (`expand` and `_expand_once`) scans a LaTeX string character by character looking for backslashes. When it finds one, it checks if the following word is a known macro. If yes — and it's not a strip target — it reads the right number of arguments (using whichever bracket style that macro expects) and substitutes them into the replacement body. It runs up to 5 passes so that macros whose bodies contain other macros get fully resolved.

In [54]:
class MacroExpander:

    def __init__(self, commands_path: str):
        self.macros: dict[str, MacroDef] = {}
        self._load(commands_path)

    # ------------------------------------------------------------------
    # Loading
    # ------------------------------------------------------------------

    def _load(self, path: str):
        with open(path, "r", encoding="utf-8") as f:
            src = f.read()

        # Strip LaTeX comments
        src = re.sub(r'%[^\n]*', '', src)

        self._parse_defs(src)
        self._parse_newcommands(src)

    def _parse_defs(self, src: str):
        """
        Match patterns:
          \\def\\name{body}
          \\def\\name#1{body}
          \\def\\name#1#2{body}
          \\def\\name[#1,#2]{body}   ← bracket-delimited args
          \\def\\name(#1,#2)(#3){body} ← paren-delimited args (quiver macros)
        """
        # Locate every \def
        i = 0
        while i < len(src):
            m = re.search(r'\\def\s*\\([A-Za-z@]+)', src[i:])
            if not m:
                break
            abs_pos = i + m.start()
            name = m.group(1)
            after_name = i + m.end()

            # Skip whitespace
            j = after_name
            while j < len(src) and src[j] in ' \t\n':
                j += 1

            # Detect argument style
            bracket_args = False
            num_args = 0

            if j < len(src) and src[j] == '[':
                # Bracket-delimited args: \def\foo[#1,#2]{...}
                bracket_content, j = _read_balanced_bracket(src, j)
                num_args = bracket_content.count('#')
                bracket_args = True
            elif j < len(src) and src[j] == '(':
                # Paren-delimited args: \def\foo(#1,#2)(#3,#4,#5){...}
                paren_args = 0
                while j < len(src) and src[j] == '(':
                    end = src.find(')', j)
                    if end == -1:
                        break
                    paren_args += src[j+1:end].count('#')
                    j = end + 1
                    while j < len(src) and src[j] in ' \t\n':
                        j += 1
                num_args = paren_args
                bracket_args = 'paren'
            else:
                # Count #1 #2 ... style args before the opening brace
                arg_section = src[j:j + 20]
                arg_matches = re.findall(r'#(\d)', arg_section)
                if arg_matches:
                    num_args = max(int(x) for x in arg_matches)
                    # Advance j past the #n tokens
                    am = re.match(r'((?:#\d\s*)+)', src[j:])
                    if am:
                        j += len(am.group(0))

            # Now read the body
            if j >= len(src) or src[j] != '{':
                i = abs_pos + 1
                continue
            try:
                body, j = _read_balanced_brace(src, j)
            except ValueError:
                i = abs_pos + 1
                continue

            strip = _is_tikz_body(body)
            self.macros[name] = MacroDef(
                name=name,
                num_args=num_args,
                replacement=body,
                bracket_args=bracket_args,
                strip=strip,
            )
            i = abs_pos + 1

    def _parse_newcommands(self, src: str):
        """
        Match:
          \\newcommand{\\name}[n]{body}
          \\newcommand{\\name}{body}
          \\renewcommand  (same)
        """
        pattern = re.compile(
            r'\\(?:new|renew)command\s*\{\\([A-Za-z@]+)\}'
            r'(?:\s*\[(\d)\])?'   # optional arg count
        )
        for m in pattern.finditer(src):
            name = m.group(1)
            num_args = int(m.group(2)) if m.group(2) else 0
            j = m.end()
            # skip whitespace
            while j < len(src) and src[j] in ' \t\n':
                j += 1
            if j >= len(src) or src[j] != '{':
                continue
            try:
                body, _ = _read_balanced_brace(src, j)
            except ValueError:
                continue
            strip = _is_tikz_body(body)
            self.macros[name] = MacroDef(
                name=name,
                num_args=num_args,
                replacement=body,
                bracket_args=False,
                strip=strip,
            )

    # ------------------------------------------------------------------
    # Expansion
    # ------------------------------------------------------------------

    def expand(self, latex: str, max_passes: int = 5) -> str:
        """
        Expand all known macros in `latex`.  Runs multiple passes so that
        macros whose bodies contain other macros are fully resolved.
        """
        result = latex
        for _ in range(max_passes):
            expanded = self._expand_once(result)
            if expanded == result:
                break
            result = expanded
        return result

    def _expand_once(self, s: str) -> str:
        """Single-pass expansion."""
        out = []
        i = 0
        while i < len(s):
            if s[i] != '\\':
                out.append(s[i])
                i += 1
                continue

            # Try to match a known macro name
            m = re.match(r'\\([A-Za-z@]+)', s[i:])
            if not m:
                out.append(s[i])
                i += 1
                continue

            name = m.group(1)
            if name not in self.macros:
                out.append(s[i:i + m.end()])
                i += m.end()
                continue

            macro = self.macros[name]
            j = i + m.end()

            if macro.strip:
                # Consume the call site and discard
                j = self._skip_macro_call(s, j, macro)
                i = j
                continue

            if macro.num_args == 0:
                out.append(macro.replacement)
                i = j
                continue

            # Collect arguments
            try:
                args, j = self._collect_args(s, j, macro)
            except ValueError:
                # Can't parse args — leave unexpanded
                out.append(s[i:i + m.end()])
                i += m.end()
                continue

            body = macro.replacement
            for idx, arg_val in enumerate(args, start=1):
                body = body.replace(f'#{idx}', arg_val)
            out.append(body)
            i = j

        return ''.join(out)

    def _collect_args(self, s: str, j: int, macro: MacroDef) -> tuple[list[str], int]:
        """Read macro arguments from position j, return (args_list, new_j)."""
        args = []
        # skip whitespace
        while j < len(s) and s[j] in ' \t\n':
            j += 1

        if macro.bracket_args == 'paren':
            # \def\foo(#1,#2)(#3,#4,#5){...} → call site is \foo(v1,v2)(v3,v4,v5)
            while j < len(s) and s[j] == '(':
                end = s.find(')', j)
                if end == -1:
                    raise ValueError("Unmatched ( in paren-arg macro call")
                group = s[j+1:end]
                args.extend(a.strip() for a in group.split(','))
                j = end + 1
                while j < len(s) and s[j] in ' \t\n':
                    j += 1
        elif macro.bracket_args:
            # \def\foo[#1,#2]{...} → call site is \foo[val1,val2]
            if j < len(s) and s[j] == '[':
                content, j = _read_balanced_bracket(s, j)
                # split on comma (naive — fine for this thesis's macros)
                args = [a.strip() for a in content.split(',')]
            else:
                raise ValueError("Expected [ for bracket-arg macro")
        else:
            # Standard {arg1}{arg2}... style
            for _ in range(macro.num_args):
                while j < len(s) and s[j] in ' \t\n':
                    j += 1
                if j < len(s) and s[j] == '{':
                    arg, j = _read_balanced_brace(s, j)
                    args.append(arg)
                elif j < len(s):
                    # Single-token argument (e.g. \macro X)
                    args.append(s[j])
                    j += 1
                else:
                    raise ValueError("Ran out of input reading macro args")

        return args, j

    def _skip_macro_call(self, s: str, j: int, macro: MacroDef) -> int:
        """Advance past a strip-target macro's argument list without emitting anything."""
        if macro.num_args == 0:
            return j
        try:
            _, j = self._collect_args(s, j, macro)
        except ValueError:
            pass
        return j

    # ------------------------------------------------------------------
    # Diagnostics
    # ------------------------------------------------------------------

    def summary(self) -> str:
        strip = [n for n, m in self.macros.items() if m.strip]
        expand = [n for n, m in self.macros.items() if not m.strip]
        lines = [
            f"MacroExpander loaded {len(self.macros)} macros",
            f"  Expand ({len(expand)}): {', '.join(sorted(expand))}",
            f"  Strip  ({len(strip)}):  {', '.join(sorted(strip))}",
        ]
        return '\n'.join(lines)

# **Block Extractor**

This file answers the question: given a chapter's LaTeX source, how do we slice it into meaningful, labelled chunks? The output is a list of Block objects — one per equation, table, caption, section heading, or paragraph of prose.

In [55]:
r"""
block_extractor.py
------------------
Walks a pylatexenc parse tree and emits a flat sequence of Block objects
for one chapter.

Block types produced:
  section_header  — \section / \subsection / \subsubsection
  equation        — display-math environments (equation, align, multline, …)
  table           — \begin{tabular} / \begin{longtable}
  caption         — \caption{} (kept even when parent figure is stripped)
  text            — paragraph prose (inline math stays embedded)

Stripped entirely:
  tikzpicture, figure (except its caption), bibliography,
  \cite, \ref, \label, \appendix preamble noise

Context windows:
  Each block gets preceding_context and following_context:
  the 2–3 sentences of plain text immediately before / after it.
"""

import re
import uuid
from dataclasses import dataclass, field
from typing import Optional

from pylatexenc.latexwalker import (
    LatexWalker,
    LatexEnvironmentNode,
    LatexMacroNode,
    LatexGroupNode,
    LatexCharsNode,
    LatexCommentNode,
    LatexMathNode,
    LatexNode,
)


def _read_balanced_brace(s: str, start: int) -> tuple[str, int]:
    """Return (content_inside_braces, index_after_closing_brace)."""
    assert s[start] == '{', f"Expected '{{' at {start}"
    depth, i = 0, start
    while i < len(s):
        if s[i] == '{':
            depth += 1
        elif s[i] == '}':
            depth -= 1
            if depth == 0:
                return s[start + 1:i], i + 1
        i += 1
    raise ValueError(f"Unbalanced braces at {start}")


### These are just sets that tell the extractor how to categorise LaTeX environments:
* `EQUATION_ENVS` — `equation`, `align`, `multline`, etc. → become equation blocks
* `TABLE_ENVS` — `tabular`, `longtable` → become table blocks
* `STRIP_ENVS` — `tikzpicture`, `figure` → thrown away (except captions)
* `STRIP_MACROS` — `\cite`, `\ref`, `\label` → thrown away
* `SECTION_MACROS` — maps `\section` → `depth 2`, `\subsection` → `depth 3`, etc.

In [56]:
# ---------------------------------------------------------------------------
# Display-math environments we extract as equation blocks
# ---------------------------------------------------------------------------
EQUATION_ENVS = {
    'equation', 'equation*',
    'align', 'align*',
    'aligned', 'aligned*',
    'multline', 'multline*',
    'gather', 'gather*',
    'eqnarray', 'eqnarray*',
    'flalign', 'flalign*',
    'subequations',
    'dmath', 'dmath*',
}

TABLE_ENVS = {'tabular', 'tabular*', 'longtable', 'array'}

STRIP_ENVS = {
    'tikzpicture',
    'tikzcd',
    'figure',
    'figure*',
    'wrapfigure',
    'thebibliography',
    'filecontents',
    'comment',
}

PASS_THROUGH_ENVS = {
    'landscape',
}

STRIP_MACROS = {
    'cite', 'citep', 'citet', 'citealt', 'citealp',
    'ref', 'eqref', 'pageref', 'autoref', 'cref', 'Cref',
    'label',
    'bibliography', 'bibliographystyle',
    'footnotemark',
    'includegraphics',
    'index',

    # page formatting
    'pagestyle',
    'thispagestyle',
    'fancyhead',
    'fancyfoot',
    'fancyhf',
    'markboth',
    'markright',
    'cleardoublepage',
}

SECTION_MACROS = {
    'chapter': 1,
    'section': 2,
    'subsection': 3,
    'subsubsection': 4,
    'paragraph': 5,
    'subparagraph':6,
}

# How many characters of surrounding text to keep as context
CONTEXT_CHARS = 600

In [57]:
# ---------------------------------------------------------------------------
# Block dataclass
# ---------------------------------------------------------------------------

@dataclass
class Block:
    block_id: str
    chapter: int
    section: str            # e.g. "3.2"
    section_title: str
    block_type: str
    content: str            # macro-expanded, human-readable
    raw_latex: str          # verbatim LaTeX
    preceding_context: str = ''
    following_context: str = ''

    def to_dict(self) -> dict:
        return {
            'block_id': self.block_id,
            'chapter': self.chapter,
            'section': self.section,
            'section_title': self.section_title,
            'block_type': self.block_type,
            'content': self.content,
            'raw_latex': self.raw_latex,
            'preceding_context': self.preceding_context,
            'following_context': self.following_context,
        }


### A small bookkeeping class that maintains the current position in the document hierarchy. Every time the extractor sees a `\section` or `\subsection`, it updates this tracker. The tracker then knows how to produce the dotted section string (e.g. "3.2.1") and the current section title. Every block gets stamped with these — that's how you'll later be able to say "this equation is from section 3.2, The Mirror Symmetry Map".

In [58]:
# ---------------------------------------------------------------------------
# Section-hierarchy tracker
# ---------------------------------------------------------------------------

class SectionTracker:
    def __init__(self):
        self._counters = [0] * 7   # index = depth (1=chapter, 2=section, …)
        self._titles = [''] * 7
        self.current_depth = 0

    def push(self, depth: int, title: str):
        """Register a new heading at `depth` (1=chapter … 5=paragraph)."""
        self._counters[depth] += 1
        # Reset deeper counters
        for d in range(depth + 1, 7):
            self._counters[d] = 0
        self._titles[depth] = title
        self.current_depth = depth

    @property
    def section_string(self) -> str:
        """Return dotted section number, e.g. '2.3.1'."""
        parts = []
        for d in range(1, self.current_depth + 1):
            if self._counters[d]:
                parts.append(str(self._counters[d]))
        return '.'.join(parts) if parts else '0'

    @property
    def section_title(self) -> str:
        return self._titles[self.current_depth] if self.current_depth else ''



### This is the main class. Its `extract()` method hands the source to `pylatexenc`, which tokenises it into a parse tree — a nested structure of nodes, one per LaTeX element. Then `_walk()` traverses that tree node by node, and `_dispatch()` decides what to do with each one:
* `LatexEnvironmentNode` (anything inside `\begin{...}\end{...}`) → `_handle_env()`
* `LatexMacroNode` (a `\command`) → `_handle_macro()`
* `LatexCharsNode` (plain text) → appended to a running text buffer
* `LatexMathNode` (`inline $...$`) → also appended to the text buffer, staying embedded in its surrounding prose

### The key design decision here is the text buffer. Prose doesn't arrive as one clean block — `pylatexenc` gives you dozens of tiny `LatexCharsNode` fragments interspersed with inline math and macros. So rather than emitting a block for every fragment, the extractor accumulates them all in `_text_buffer`. Whenever something structurally significant arrives — a section heading, a display equation, a table — it first calls `_flush_text_block()`, which drains the buffer into a single text block, then handles the new element.

### For figures specifically: the entire figure environment is stripped, but before discarding it the extractor walks its children looking for `\caption` nodes and emits those as caption blocks. That way you keep the descriptive text even though the actual drawing is gone.

In [59]:
# ---------------------------------------------------------------------------
# Main extractor
# ---------------------------------------------------------------------------

class BlockExtractor:

    def __init__(self, chapter: int, expander: MacroExpander):
        self.chapter = chapter
        self.expander = expander
        self.tracker = SectionTracker()
        self._blocks: list[Block] = []
        self._block_counter = 0
        # Running plain-text buffer for context extraction
        self._text_buffer: list[str] = []   # (block_index_or_None, text_snippet)

    # ------------------------------------------------------------------
    # Public entry point
    # ------------------------------------------------------------------

    def extract(self, latex_source: str) -> list[Block]:
        """Parse `latex_source` and return list of Block objects."""
        walker = LatexWalker(latex_source, tolerant_parsing=True)
        nodelist, _, _ = walker.get_latex_nodes()
        self._walk(nodelist, latex_source)
        self._flush_text_block()   # flush any trailing text not followed by a structural element
        self._attach_contexts(latex_source)
        return self._blocks

    # ------------------------------------------------------------------
    # Tree walker
    # ------------------------------------------------------------------

    def _walk(self, nodelist, source: str):
        if nodelist is None:
            return
        for node in nodelist:
            self._dispatch(node, source)

    def _dispatch(self, node, source: str):
        if node is None:
            return

        if isinstance(node, LatexCommentNode):
            return

        if isinstance(node, LatexEnvironmentNode):
            self._handle_env(node, source)

        elif isinstance(node, LatexMacroNode):
            self._handle_macro(node, source)

        elif isinstance(node, LatexMathNode):
            # Inline math — keep as part of surrounding text, don't emit a block
            self._text_buffer.append(('inline_math', node.latex_verbatim()))

        elif isinstance(node, LatexCharsNode):
            text = node.chars
            if text.strip():
                self._text_buffer.append(('text', text))

        elif isinstance(node, LatexGroupNode):
            self._walk(node.nodelist, source)

        elif hasattr(node, 'nodelist') and node.nodelist:
            self._walk(node.nodelist, source)

    def _handle_env(self, node: LatexEnvironmentNode, source: str):
        env = node.environmentname

        if env in STRIP_ENVS:
            if env in ('figure', 'figure*', 'wrapfigure'):
                # Extract captions before stripping
                self._extract_captions_from_figure(node, source)
            return  # strip

        if env in EQUATION_ENVS:
            self._emit_equation(node, source)
            return

        if env in TABLE_ENVS:
            self._emit_table(node, source)
            return

        if env in PASS_THROUGH_ENVS:
            if node.nodelist:
              self._walk(node.nodelist, source)
            return

        if env == "landscape":
            self._extract_captions_from_figure(node, source)
            self._walk(node.nodelist, source)
            return

        # Recurse into other environments (theorem, proof, itemize, etc.)
        if node.nodelist:
            self._walk(node.nodelist, source)

    def _handle_macro(self, node: LatexMacroNode, source: str):
        name = node.macroname

        if name.endswith('*'):
          name = name[:-1]

        if name in STRIP_MACROS:
            return

        if name in SECTION_MACROS:
            self._flush_text_block()
            self._handle_section(node, source)
            return

        if name in ('footnote', 'footnotetext'):
            self._handle_footnote(node, source)
            return

        if name == 'caption':
            self._emit_caption(node, source)
            return

        # For other macros, try to expand and add to text buffer
        raw = node.latex_verbatim()
        expanded = self.expander.expand(raw)
        if expanded.strip():
            self._text_buffer.append(('text', expanded))

    # ------------------------------------------------------------------
    # Footnote handling
    # ------------------------------------------------------------------

    def _handle_footnote(self, node: LatexMacroNode, source: str):
        """
        Extract footnote text and append it to the running text buffer,
        wrapped in [Footnote: ...] so downstream generators know its role.
        Handles both \footnote{text} and \footnotetext{text}.
        """
        footnote_text = ''

        # Try parsed args first
        if node.nodeargd and node.nodeargd.argnlist:
            for arg in node.nodeargd.argnlist:
                if arg:
                    footnote_text = self._node_to_text(arg)
                    break

        # Fallback: extract from raw latex via brace matching
        if not footnote_text.strip():
            raw = node.latex_verbatim()
            m = re.search(r'\\footnote(?:text)?\s*\{', raw)
            if m:
                try:
                    footnote_text, _ = _read_balanced_brace(raw, m.end() - 1)
                except (ValueError, AssertionError):
                    pass

        if footnote_text.strip():
            expanded = self.expander.expand(footnote_text)
            self._text_buffer.append(('footnote', f'[Footnote: {expanded.strip()}]'))

    # ------------------------------------------------------------------
    # Section handling
    # ------------------------------------------------------------------

    def _handle_section(self, node: LatexMacroNode, source: str):
        depth = SECTION_MACROS[node.macroname]

        # Extract title from first argument group
        title = ''
        if node.nodeargd and node.nodeargd.argnlist:
            for arg in node.nodeargd.argnlist:
                if arg and isinstance(arg, LatexGroupNode):
                    title = self._node_to_text(arg)
                    break

        self.tracker.push(depth, title)

        raw = node.latex_verbatim()
        block = self._make_block(
            block_type='section_header',
            content=title,
            raw_latex=raw,
        )
        self._blocks.append(block)

    # ------------------------------------------------------------------
    # Equation blocks
    # ------------------------------------------------------------------

    def _equation_is_figure(self, raw: str) -> bool:
        signals = (
            r'\includegraphics',
            r'\begin{tikzpicture}',
            r'\tikz{',
            r'\begin{tikzcd}',
            )
        return any(sig in raw for sig in signals)

    def _emit_equation(self, node: LatexEnvironmentNode, source: str):

        raw = node.latex_verbatim()
        if self._equation_is_figure(raw):
            return

        self._flush_text_block()
        raw = node.latex_verbatim()
        # content: expand macros but keep the math structure readable
        content = self.expander.expand(raw)
        block = self._make_block(
            block_type='equation',
            content=content,
            raw_latex=raw,
        )
        self._blocks.append(block)

    # ------------------------------------------------------------------
    # Table blocks
    # ------------------------------------------------------------------

    def _emit_table(self, node: LatexEnvironmentNode, source: str):
        self._flush_text_block()
        raw = node.latex_verbatim()
        content = self.expander.expand(raw)
        block = self._make_block(
            block_type='table',
            content=content,
            raw_latex=raw,
        )
        self._blocks.append(block)

    # ------------------------------------------------------------------
    # Caption blocks
    # ------------------------------------------------------------------

    def _emit_caption(self, node: LatexMacroNode, source: str,caption_group=None,):
        original_raw = node.latex_verbatim()
        caption_text = ""

        if caption_group is not None:
            caption_text = self._node_to_text(caption_group)

        if not caption_text.strip():
            if node.nodeargd and node.nodeargd.argnlist:
              for arg in node.nodeargd.argnlist:
                if arg:
                  caption_text = self._node_to_text(arg)
                  break

        # Fallback: extract from local raw latex
        if not caption_text.strip():
            m = re.search(r'\\caption\s*\{', original_raw)
            if m:
                try:
                    caption_text, _ = _read_balanced_brace(
                        original_raw,
                        m.end() - 1
                        )
                except (ValueError, AssertionError):
                    pass

        # Final fallback
        if not caption_text.strip():
            caption_text = original_raw

        content = self.expander.expand(caption_text)

        # Store useful caption source instead of bare "\caption"
        raw_latex = f"\\caption{{{caption_text}}}"

        block = self._make_block(
            block_type='caption',
            content=content,
            raw_latex=raw_latex,
        )

        self._blocks.append(block)

    def _extract_captions_from_figure(self, node, source):
        children = node.nodelist or []

        for i, child in enumerate(children):

            if (isinstance(child, LatexMacroNode) and child.macroname == "caption"):
                caption_group = None
                if (i + 1 < len(children) and isinstance(children[i + 1], LatexGroupNode)):
                    caption_group = children[i + 1]

                self._emit_caption(child, source, caption_group)


    #def _extract_captions_from_figure(self, node: LatexEnvironmentNode, source: str):
       # r"""Walk a figure environment and emit only its \caption nodes."""
        #for caption in self._find_captions(node):
          #self._emit_caption(caption, source)


    def _find_captions(self, node):
        found = []

        if isinstance(node, LatexMacroNode):
            if node.macroname == "caption":
                found.append(node)

        if hasattr(node, "nodelist") and node.nodelist:
            for child in node.nodelist:
                found.extend(self._find_captions(child))

        return found
    # ------------------------------------------------------------------
    # Text block flushing
    # ------------------------------------------------------------------

    def _flush_text_block(self):
        """Drain _text_buffer into a text Block if it has meaningful content."""
        parts = [t for _, t in self._text_buffer]
        combined = ' '.join(parts).strip()
        combined = re.sub(r'\s+', ' ', combined)

        # Only emit if there's a meaningful amount of prose
        if len(combined) < 20:
            self._text_buffer.clear()
            return

        content = self.expander.expand(combined)
        raw = combined  # already extracted text, not raw LaTeX here

        block = self._make_block(
            block_type='text',
            content=content,
            raw_latex=raw,
        )
        self._blocks.append(block)
        self._text_buffer.clear()

    # ------------------------------------------------------------------
    # Block factory
    # ------------------------------------------------------------------

    def _make_block(self, block_type: str, content: str, raw_latex: str) -> Block:
        self._block_counter += 1
        block_id = (
            f"ch{self.chapter}"
            f"_s{self.tracker.section_string.replace('.', '_')}"
            f"_block_{self._block_counter:03d}"
        )
        return Block(
            block_id=block_id,
            chapter=self.chapter,
            section=self.tracker.section_string,
            section_title=self.tracker.section_title,
            block_type=block_type,
            content=content.strip(),
            raw_latex=raw_latex.strip(),
        )

    # ------------------------------------------------------------------
    # Context attachment (post-pass)
    # ------------------------------------------------------------------

    def _attach_contexts(self, source: str):
        r"""
        For each block, find its raw_latex in the source and attach
        the surrounding plain-text sentences as context.
        """
        # Build a plain-text version of source for context extraction
        plain = _latex_to_plain(source)

        for i, block in enumerate(self._blocks):
            if not block.raw_latex:
                continue

            # Find approximate char position of this block in source
            pos = source.find(block.raw_latex[:60])  # use first 60 chars as anchor
            if pos == -1:
                continue

            # Map source position to plain-text position (rough)
            plain_pos = int(pos * len(plain) / max(len(source), 1))

            pre_start = max(0, plain_pos - CONTEXT_CHARS)
            post_end = min(len(plain), plain_pos + len(block.raw_latex) + CONTEXT_CHARS)

            block.preceding_context = _last_sentences(plain[pre_start:plain_pos], n=3)
            block.following_context = _first_sentences(plain[plain_pos:post_end], n=3)

    # ------------------------------------------------------------------
    # Utility
    # ------------------------------------------------------------------

    def _node_to_text(self, node) -> str:
        """Recursively extract plain text from a node."""
        if node is None:
            return ''
        if isinstance(node, LatexCharsNode):
            return node.chars
        if isinstance(node, LatexGroupNode):
            return ''.join(self._node_to_text(c) for c in (node.nodelist or []))
        if isinstance(node, LatexMacroNode):
            raw = node.latex_verbatim()
            return self.expander.expand(raw)
        if isinstance(node, LatexMathNode):
            return node.latex_verbatim()
        if hasattr(node, 'nodelist') and node.nodelist:
            return ''.join(self._node_to_text(c) for c in node.nodelist)
        return ''

In [60]:
# ---------------------------------------------------------------------------
# Plain-text helpers for context
# ---------------------------------------------------------------------------

def _latex_to_plain(latex: str) -> str:
    """Very lightweight LaTeX → plain text for context extraction."""
    # Strip comments
    s = re.sub(r'%[^\n]*', '', latex)
    # Strip common markup
    s = re.sub(r'\\(?:textbf|emph|textit|text|mathrm|mathbf)\{([^}]*)\}', r'\1', s)
    # Remove remaining macros with args
    s = re.sub(r'\\[A-Za-z]+\{[^}]*\}', '', s)
    # Remove standalone macros
    s = re.sub(r'\\[A-Za-z]+', ' ', s)
    # Remove math delimiters
    s = re.sub(r'\$\$.*?\$\$', ' [equation] ', s, flags=re.DOTALL)
    s = re.sub(r'\$[^$]*\$', ' [math] ', s)
    # Collapse braces
    s = re.sub(r'[{}]', '', s)
    # Collapse whitespace
    s = re.sub(r'\s+', ' ', s)
    return s.strip()


def _sentence_split(text: str) -> list[str]:
    return re.split(r'(?<=[.!?])\s+', text.strip())


def _last_sentences(text: str, n: int = 3) -> str:
    sentences = _sentence_split(text)
    return ' '.join(sentences[-n:]).strip()


def _first_sentences(text: str, n: int = 3) -> str:
    sentences = _sentence_split(text)
    return ' '.join(sentences[:n]).strip()


# **LatexParser**

This is the entry point that ties everything together. It has two parts.
`FileResolver` handles the fact that a thesis isn't one file — it's a main.tex that `\inputs` chapter files, which may themselves `\input` sub-files. The resolver reads `main.tex`, extracts the `\begin{document}...\end{document}` body, finds every `\input{}`/`\include{}` call in order, and recursively inlines nested includes. The result is a flat list of (`chapter_number`, `filename_stem`, `full_source_string`) tuples — one per chapter — with all sub-files already merged in. It skips includes whose filenames contain preamble, style, etc., since those are setup files, not content.

`ThesisParser` then loops over those tuples. For each chapter it creates a fresh `BlockExtractor`, runs it, and writes the blocks to a per-chapter JSON file (`ch01_introduction.json`, etc.) plus a combined `all_blocks`.json. At the end it prints a summary table — block counts by type and by chapter — so you can immediately spot if a chapter produced suspiciously few blocks.
There's also a `parse_single_file()` convenience function for testing one chapter in isolation without needing a full main.tex.

In [61]:
"""
latex_parser.py
---------------

Responsibilities:
  1. Resolve the multi-file thesis structure via \\input / \\include
  2. Run MacroExpander + BlockExtractor on each chapter
  3. Write per-chapter block JSON to data/blocks/

Usage (CLI):
    python latex_parser.py \\
        --root    data/raw/main.tex \\
        --macros  config/custom_commands.tex \\
        --outdir  data/blocks/

Usage (import):
    from latex_parser import ThesisParser
    parser = ThesisParser(root_tex, macros_tex, outdir)
    all_blocks = parser.run()
"""

import argparse
import json
import logging
import os
import re
import sys
from pathlib import Path
from typing import Optional


logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')
log = logging.getLogger(__name__)


In [62]:
# ---------------------------------------------------------------------------
# Multi-file resolver
# ---------------------------------------------------------------------------

class FileResolver:
    """
    Resolves \\input{...} and \\include{...} directives, returning a flat
    list of (chapter_number, resolved_source) pairs.

    Chapters are identified as top-level \\input/\\include calls in the main
    file whose filenames contain 'ch', 'chapter', or 'app' (for appendices),
    or simply in document order.
    """

    def __init__(self, root: Path):
        self.root = root
        self.base_dir = root.parent

    def resolve_all(self) -> list[tuple[int, str, str]]:
        """
        Returns list of (chapter_number, filename_stem, source_string).
        chapter_number is 1-based.
        """
        root_source = self._read(self.root)
        # Extract the \begin{document}...\end{document} body
        body = self._extract_document_body(root_source)

        # Find all top-level \input / \include lines
        includes = self._find_includes(body)
        log.info(f"Found {len(includes)} top-level includes in {self.root.name}")

        chapters = []
        ch_num = 0
        for inc_path in includes:
            full_path = self._locate(inc_path)
            if full_path is None:
                log.warning(f"Could not locate: {inc_path}")
                continue
            ch_num += 1
            source = self._read_recursive(full_path)
            stem = full_path.stem
            log.info(f"  Chapter {ch_num}: {stem} ({len(source):,} chars)")
            chapters.append((ch_num, stem, source))

        return chapters

    def _extract_document_body(self, source: str) -> str:
        m = re.search(
            r'\\begin\s*\{document\}(.*?)\\end\s*\{document\}',
            source, re.DOTALL
        )
        return m.group(1) if m else source

    def _find_includes(self, source: str) -> list[str]:
        """Return file paths from \\input{} and \\include{} (order preserved)."""
        paths = []
        for m in re.finditer(r'\\(?:input|include)\s*\{([^}]+)\}', source):
            p = m.group(1).strip()
            # Ignore includes of preamble/style files
            if any(skip in p.lower() for skip in ('preamble', 'style', 'command', 'package')):
                log.debug(f"Skipping preamble include: {p}")
                continue
            paths.append(p)
        return paths

    def _locate(self, path_str: str) -> Optional[Path]:
        """Try to find the file, appending .tex if needed."""
        candidates = [
            self.base_dir / path_str,
            self.base_dir / (path_str + '.tex'),
        ]
        for c in candidates:
            if c.exists():
                return c
        return None

    def _read(self, path: Path) -> str:
        with open(path, 'r', encoding='utf-8', errors='replace') as f:
            return f.read()

    def _read_recursive(self, path: Path) -> str:
        """Read a file, inlining any nested \\input/\\include calls."""
        source = self._read(path)
        # Strip LaTeX comments before resolving nested includes
        source_no_comments = re.sub(r'%[^\n]*', '', source)

        def replacer(m):
            nested = m.group(1).strip()
            nested_path = path.parent / nested
            if not nested_path.suffix:
                nested_path = nested_path.with_suffix('.tex')
            if nested_path.exists():
                log.debug(f"  Inlining nested include: {nested_path.name}")
                return self._read_recursive(nested_path)
            return m.group(0)  # leave unresolved

        resolved = re.sub(r'\\(?:input|include)\s*\{([^}]+)\}', replacer, source)
        return resolved


In [63]:
class ThesisParser:

    def __init__(self, root_tex: str, macros_tex: str, outdir: str):
        self.root = Path(root_tex)
        self.macros = Path(macros_tex)
        self.outdir = Path(outdir)
        self.outdir.mkdir(parents=True, exist_ok=True)

    def _preclean_latex(self, tex:str) -> str:
      # ---------------------------
      # page layout junk
      # ---------------------------
      tex = re.sub(r'\\setlength\s*\{\\headheight\}\s*\{[^}]*\}', '', tex)
      tex = re.sub(r'\\pagestyle\s*\{[^}]*\}', '', tex)
      tex = re.sub(r'\\fancy(head|foot|hf)\s*(\[[^\]]*\])?\s*\{[^}]*\}', '', tex)

      tex = re.sub(r'\[(LE|RO|LO|RE)(\s*,\s*(LE|RO|LO|RE))*\]', '', tex)

      return tex

    def run(self) -> list[dict]:
        """Run Stage 1 end-to-end. Returns all blocks as list of dicts."""
        # 1. Load macro expander
        log.info(f"Loading macros from {self.macros}")
        expander = MacroExpander(str(self.macros))
        log.info(expander.summary())

        # 2. Resolve files
        resolver = FileResolver(self.root)
        chapters = resolver.resolve_all()

        if not chapters:
            log.error("No chapters found. Check your root .tex file.")
            return []

        # 3. Extract blocks per chapter
        all_blocks = []
        for ch_num, stem, source in chapters:
            log.info(f"Extracting blocks from chapter {ch_num}: {stem}")
            source = self._preclean_latex(source)
            extractor = BlockExtractor(chapter=ch_num, expander=expander)
            blocks = extractor.extract(source)
            block_dicts = [b.to_dict() for b in blocks]
            all_blocks.extend(block_dicts)

            # Write per-chapter JSON
            out_path = self.outdir / f"ch{ch_num:02d}_{stem}.json"
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump(block_dicts, f, indent=2, ensure_ascii=False)
            log.info(f"  → {len(blocks)} blocks written to {out_path.name}")

        # 4. Write combined JSON
        combined_path = self.outdir / 'all_blocks.json'
        with open(combined_path, 'w', encoding='utf-8') as f:
            json.dump(all_blocks, f, indent=2, ensure_ascii=False)
        log.info(f"Combined: {len(all_blocks)} blocks → {combined_path}")

        # 5. Print summary
        self._print_summary(all_blocks)
        return all_blocks

    def _print_summary(self, blocks: list[dict]):
        from collections import Counter
        type_counts = Counter(b['block_type'] for b in blocks)
        ch_counts = Counter(b['chapter'] for b in blocks)
        print("\n=== Stage 1 Summary ===")
        print(f"Total blocks: {len(blocks)}")
        print("By type:")
        for t, n in sorted(type_counts.items()):
            print(f"  {t:20s} {n:4d}")
        print("By chapter:")
        for ch, n in sorted(ch_counts.items()):
            print(f"  Chapter {ch}:           {n:4d}")


In [64]:
# ---------------------------------------------------------------------------
# Single-file mode (for testing on one .tex without a root file)
# ---------------------------------------------------------------------------

def parse_single_file(
    tex_path: str,
    macros_path: str,
    chapter_num: int = 1,
    outdir: str = 'data/blocks',
) -> list[dict]:
    """
    Parse a single .tex file directly — useful for testing one chapter
    without a full thesis root file.
    """
    outdir_path = Path(outdir)
    outdir_path.mkdir(parents=True, exist_ok=True)

    expander = MacroExpander(macros_path)
    log.info(expander.summary())

    with open(tex_path, 'r', encoding='utf-8', errors='replace') as f:
        source = f.read()

    extractor = BlockExtractor(chapter=chapter_num, expander=expander)
    blocks = extractor.extract(source)
    block_dicts = [b.to_dict() for b in blocks]

    stem = Path(tex_path).stem
    out_path = outdir_path / f"ch{chapter_num:02d}_{stem}.json"
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(block_dicts, f, indent=2, ensure_ascii=False)

    log.info(f"{len(blocks)} blocks written to {out_path}")
    return block_dicts

# Sanity Checks

In [48]:
blocks = parse_single_file(
    tex_path   = '/content/drive/MyDrive/SISSA_Thesis/chapters/Unitary_Mirrors_1.tex',
    macros_path= '/content/drive/MyDrive/SISSA_Thesis/custom_commands.tex',
    chapter_num= 2,
    outdir     = '/content/drive/MyDrive/SISSA_Thesis/sample_blocks'
)


In [67]:
parser = ThesisParser(
    root_tex  = '/content/drive/MyDrive/SISSA_Thesis/main.tex',
    macros_tex= '/content/drive/MyDrive/SISSA_Thesis/custom_commands.tex',
    outdir    = '/content/drive/MyDrive/SISSA_Thesis/blocks_a'
)
parser.run()


=== Stage 1 Summary ===
Total blocks: 1188
By type:
  caption                73
  equation              354
  section_header        201
  table                  19
  text                  541
By chapter:
  Chapter 1:            338
  Chapter 2:            280
  Chapter 3:            141
  Chapter 4:            136
  Chapter 5:            230
  Chapter 6:             12
  Chapter 7:             16
  Chapter 8:             35


[{'block_id': 'ch1_s1_block_001',
  'chapter': 1,
  'section': '1',
  'section_title': 'Geometric Engineering of Three-Dimensional Supersymmetric Gauge Theories',
  'block_type': 'section_header',
  'content': 'Geometric Engineering of Three-Dimensional Supersymmetric Gauge Theories',
  'raw_latex': '\\chapter{Geometric Engineering of Three-Dimensional Supersymmetric Gauge Theories}',
  'preceding_context': 'Su',
  'following_context': 'persymmetry has played a central role in shaping our modern understanding of quantum field theory beyond perturbation theory. By severely constraining the structure of quantum corrections, supersymmetric theories admit exact results that would otherwise be inaccessible, allowing for the controlled study of strongly coupled dynamics. Among the most striking consequences of these constraints is the existence of nonperturbative dualities: distinct microscopic descriptions that flow to the same infrared fixed point.'},
 {'block_id': 'ch1_s1_block_002',
  'c